# San Francisco Board of Supervisors Meeting Minutes Chatbot

This notebook demonstrates an automated pipeline for downloading, processing, and analyzing San Francisco Board of Supervisors meeting minutes using Snowflake's advanced unstructured data capabilities.

## 🎯 Objectives
- **Automated Data Collection**: Scrape and download meeting minutes from the SF BOS website
- **Document Processing**: Parse PDF documents using Snowflake's AI-powered document parsing
- **Intelligent Search**: Create a searchable knowledge base using Cortex Search
- **Data Analysis**: Enable semantic search and analysis of meeting content
---


In [ ]:
import streamlit as st
st.image('sfbos_rag.png')

## 🏗️ Architecture Overview
1. **Data Ingestion**: Python-based web scraping to collect meeting minutes
2. **Storage**: Snowflake internal stages for secure document storage
3. **Processing**: AI-powered document parsing and text chunking
4. **Search**: Cortex Search service for semantic document retrieval
---

## 📊 Database and Schema Setup

Let's start by setting up our database infrastructure for storing and processing the meeting minutes.


In [ ]:
USE ROLE SYSADMIN;

In [ ]:
CREATE DATABASE IF NOT EXISTS ADVANCED_ANALYTICS;

In [ ]:
CREATE SCHEMA IF NOT EXISTS ADVANCED_ANALYTICS.UNSTRUCTURED;

### 📁 Storage Setup

Create a Snowflake stage for storing the downloaded PDF documents with directory table functionality enabled for easy file management.

Note that we are enabling a directory table, and using the Server-Side encryption option. Both are required for the current project.


In [ ]:
CREATE OR REPLACE STAGE ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES 
    DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE) 
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE') 
    COMMENT = 'Stage for San Francisco Board of Supervisors meeting minutes';

### 📋 Metadata Tracking

Create a metadata table to track document information, URLs, and meeting dates extracted from filenames.


In [ ]:
-- Create metadata table to track document URLs and details
CREATE OR REPLACE TABLE ADVANCED_ANALYTICS.UNSTRUCTURED.MINUTES_METADATA (
    ID NUMBER AUTOINCREMENT PRIMARY KEY,
    RELATIVE_PATH VARCHAR(500) NOT NULL,
    DOCUMENT_URL VARCHAR(1000) NOT NULL,
    DOCUMENT_NAME VARCHAR(500),
    MEETING_DATE DATE,
    DOWNLOAD_TIMESTAMP TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

## 🤖 Automated Document Collection

This stored procedure automatically scrapes the SF Board of Supervisors website, downloads meeting minutes PDFs, and extracts meeting dates from filenames.

### Prerequisite: 
You must first set up a Network Rule and an External Access Integration (EAI) to access SF BOS website. For example:
```sql
--EAI must be created with an Accountadmin role (or a role with a grant to create EAI on account)
USE ROLE ACCOUNTADMIN;

-- Create a network rule to allow access to the SFBOS website
CREATE OR REPLACE NETWORK RULE ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_NETWORK_RULE
    MODE = EGRESS
    TYPE = HOST_PORT
    VALUE_LIST = ('sfbos.org');

-- Create external access integration
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION SFBOS_API_INTEGRATION
    ALLOWED_NETWORK_RULES = (ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_NETWORK_RULE)
    ENABLED = true;

GRANT USAGE ON INTEGRATION SFBOS_API_INTEGRATION TO ROLE SYSADMIN;
```


In [ ]:
-- Enhanced SF BOS Minutes Procedure with Improved Date Parsing
-- Extracts meeting dates from filename patterns (bagMMDDYY.pdf)
CREATE PROCEDURE IF NOT EXISTS ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES_DOWNLOAD()
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.12'
PACKAGES = ('snowflake-snowpark-python','beautifulsoup4','requests')
HANDLER = 'working_sfbos_minutes'
EXTERNAL_ACCESS_INTEGRATIONS = (SFBOS_API_INTEGRATION)
EXECUTE AS OWNER
AS '
import os
import requests
from bs4 import BeautifulSoup
import re
import time
from snowflake.snowpark import functions as F
from datetime import datetime
from urllib.parse import urljoin, urlparse

def working_sfbos_minutes(session):

    # Base URL for SFBOS Full Board Meetings
    base_url = "https://sfbos.org/meetings/full-board-meetings"
    
    # Directory to save downloaded minutes temporarily
    save_dir = "/tmp/"
    total_downloaded = 0
    page_num = 0
    max_pages = 20  # Safety limit to prevent infinite loop
    
    print("Starting to scrape SF BOS minutes using enhanced approach from " + base_url)

    while page_num < max_pages:
        try:
            # Construct URL for current page
            if page_num == 0:
                page_url = base_url
            else:
                page_url = base_url + "?page=" + str(page_num)
            
            print("\\nProcessing page " + str(page_num) + ": " + page_url)
            
            # Fetch the page
            response = requests.get(page_url, timeout=30)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, "html.parser")
            
            # Find minutes links using the working approach
            minutes_links = []
            
            # Loop through the table rows and extract minutes links
            for row in soup.find_all("tr"):
                # Get the content of the first cell (which should contain the Date)
                date_cell = row.find("td")
                if not date_cell:
                    continue
                
                date_info = date_cell.get_text(strip=True)
                
                # Look for links containing the word "Minutes" (case-insensitive)
                link = row.find("a", string=re.compile(r"Minutes", re.IGNORECASE))
                if link:
                    href = link.get("href")
                    full_url = urljoin(base_url, href)  # Join base URL with relative URL
                    minutes_links.append({
                        "minutes_url": full_url,
                        "date_info": date_info,
                        "href": href,
                        "text": link.get_text(strip=True)
                    })
            
            print("Found " + str(len(minutes_links)) + " minutes links on page " + str(page_num))
            
            # If no meetings found on this page, we''ve reached the end
            if len(minutes_links) == 0:
                print("No more meetings found. Stopping at page " + str(page_num))
                break
            
            # Process each minutes link
            for minute_info in minutes_links:
                try:
                    minutes_url = minute_info["minutes_url"]
                    date_info = minute_info["date_info"]
                    
                    print("\\nAttempting to download minutes from: " + minutes_url)
                    
                    # Make a request to the minutes link (which should redirect to PDF)
                    minutes_response = requests.get(minutes_url, timeout=30, allow_redirects=True, stream=True)
                    minutes_response.raise_for_status()
                    
                    # Check the final URL after redirects
                    final_url = minutes_response.url
                    print("Final URL after redirects: " + final_url)
                    
                    if final_url.lower().endswith(".pdf"):
                        # Direct PDF redirect - this is what we expect
                        pdf_content = minutes_response.content
                        
                        # Get the file name from the final URL
                        file_name = os.path.basename(urlparse(final_url).path)
                        
                        if not file_name or not file_name.lower().endswith(".pdf"):
                            # Generate filename from href if needed
                            href_filename = minute_info["href"].split("/")[-1] + ".pdf"
                            file_name = href_filename.replace("%5F", "_")  # URL decode underscores
                        
                        minutes_path = os.path.join(save_dir, file_name)
                        
                        # Save the PDF locally
                        with open(minutes_path, "wb") as f:
                            f.write(pdf_content)
                        print("Successfully saved PDF: " + file_name)
                        
                        # Upload to Snowflake stage
                        session.file.put(minutes_path, "@ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES", auto_compress=False, overwrite=True)
                        print("Uploaded " + file_name + " to Snowflake stage")
                        
                        # Get the relative path of the uploaded file (following working example pattern)
                        file_list = session.sql("LIST @ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES").collect()
                        relative_path = None
                        for row in file_list:
                            if row[0].endswith(file_name): # Assuming the name is unique enough
                                relative_path = row[0]
                                break
                        
                        if relative_path:
                            # Extract meeting date from filename (pattern: bagMMDDYY.pdf)
                            meeting_date = None
                            
                            # Try to extract date from filename first
                            filename_without_ext = file_name.replace(".pdf", "").replace(".PDF", "")
                            
                            # Look for pattern: bag followed by 6 digits (MMDDYY)
                            filename_date_match = re.search(r"bag(\\d{6})", filename_without_ext, re.IGNORECASE)
                            
                            if filename_date_match:
                                try:
                                    date_str = filename_date_match.group(1)  # Extract the 6 digits
                                    
                                    # Parse as MMDDYY format (e.g., 110425 = 11/04/25 = November 4, 2025)
                                    month = int(date_str[:2])
                                    day = int(date_str[2:4])
                                    year_two_digit = int(date_str[4:6])
                                    
                                    # Convert 2-digit year to 4-digit year (assume 2000s)
                                    year = 2000 + year_two_digit
                                    
                                    # Validate month and day ranges
                                    if 1 <= month <= 12 and 1 <= day <= 31:
                                        meeting_date = str(year) + "-" + str(month).zfill(2) + "-" + str(day).zfill(2)
                                        print("Extracted meeting date from filename " + file_name + ": " + meeting_date)
                                    else:
                                        print("Invalid date components in filename " + file_name + ": month=" + str(month) + ", day=" + str(day) + ", year=" + str(year))
                                        
                                except ValueError as e:
                                    print("Error parsing date from filename " + file_name + ": " + str(e))
                            else:
                                print("Could not extract date from filename pattern: " + file_name)
                            
                            # Fallback: try to extract from date_info if filename parsing failed
                            if not meeting_date and date_info:
                                print("Attempting fallback date parsing from webpage date_info: " + date_info)
                                # Try simple date patterns as fallback
                                date_patterns = [
                                    r"(\\d{1,2})[/-](\\d{1,2})[/-](\\d{4})",         # MM/DD/YYYY
                                    r"(\\d{4})[/-](\\d{1,2})[/-](\\d{1,2})",         # YYYY/MM/DD
                                ]
                                
                                for pattern in date_patterns:
                                    match = re.search(pattern, date_info)
                                    if match:
                                        try:
                                            groups = match.groups()
                                            if pattern.startswith(r"(\\d{4})"):  # YYYY first
                                                meeting_date = str(int(groups[0])) + "-" + str(int(groups[1])).zfill(2) + "-" + str(int(groups[2])).zfill(2)
                                                break
                                            else:  # MM first
                                                meeting_date = str(int(groups[2])) + "-" + str(int(groups[0])).zfill(2) + "-" + str(int(groups[1])).zfill(2)
                                                break
                                        except:
                                            continue
                            
                            # Insert metadata into the table using simple approach
                            try:
                                if meeting_date:
                                    session.sql("INSERT INTO ADVANCED_ANALYTICS.UNSTRUCTURED.MINUTES_METADATA (RELATIVE_PATH, DOCUMENT_URL, DOCUMENT_NAME, MEETING_DATE, DOWNLOAD_TIMESTAMP) VALUES (?, ?, ?, ?, CURRENT_TIMESTAMP())", 
                                               [relative_path, final_url, file_name, meeting_date]).collect()
                                    print("Metadata inserted for " + file_name + " with meeting date " + meeting_date)
                                else:
                                    session.sql("INSERT INTO ADVANCED_ANALYTICS.UNSTRUCTURED.MINUTES_METADATA (RELATIVE_PATH, DOCUMENT_URL, DOCUMENT_NAME, DOWNLOAD_TIMESTAMP) VALUES (?, ?, ?, CURRENT_TIMESTAMP())", 
                                               [relative_path, final_url, file_name]).collect()
                                    print("Metadata inserted for " + file_name + " (no date extracted)")
                            except Exception as metadata_error:
                                print("Error inserting metadata for " + file_name + ": " + str(metadata_error))
                                # Continue processing other files even if metadata fails
                            
                            print("Successfully processed file: " + file_name)
                            print("  - Stage path: " + relative_path)
                            print("  - Original URL: " + final_url)
                            if meeting_date:
                                print("  - Meeting date: " + meeting_date)
                        else:
                            print("Could not determine relative path for " + file_name)
                        
                        # Remove the local file after upload
                        os.remove(minutes_path)
                        total_downloaded += 1
                        
                    else:
                        print("URL does not point to a PDF: " + final_url)
                
                except requests.exceptions.RequestException as e:
                    print("Error processing minutes link " + minute_info["minutes_url"] + ": " + str(e))
                except Exception as e:
                    print("Unexpected error processing minutes link " + minute_info["minutes_url"] + ": " + str(e))
            
            # Move to next page
            page_num += 1
            
            # Small delay between pages to be respectful to the server
            time.sleep(1)
            
        except requests.exceptions.RequestException as e:
            print("Error accessing page " + str(page_num) + ": " + str(e))
            break
        except Exception as e:
            print("Unexpected error processing page " + str(page_num) + ": " + str(e))
            break
    
    return "Successfully processed " + str(page_num + 1) + " pages and uploaded " + str(total_downloaded) + " board minutes documents with enhanced date parsing."

';


### 🚀 Execute Data Collection

Run the enhanced procedure to download and process meeting minutes with improved date extraction from filenames.

In [ ]:
call ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES_DOWNLOAD();

## 📊 Data Verification and Analysis

Let's examine the uploaded files and verify that our enhanced date parsing is working correctly.


In [ ]:
LIST @ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES;

In [ ]:
-- Query metadata table to see downloaded files with URLs and details
SELECT 
    ID,
    DOCUMENT_NAME,
    DOCUMENT_URL,
    MEETING_DATE,
    DOWNLOAD_TIMESTAMP,
    RELATIVE_PATH
FROM ADVANCED_ANALYTICS.UNSTRUCTURED.MINUTES_METADATA 
ORDER BY DOWNLOAD_TIMESTAMP DESC
LIMIT 10;



In [ ]:
select * from ADVANCED_ANALYTICS.UNSTRUCTURED.MINUTES_METADATA ;

In [ ]:
-- Summary statistics of downloaded minutes
SELECT 
    'SUMMARY STATISTICS' AS INFO,
    COUNT(*) as TOTAL_DOCUMENTS,
    COUNT(DISTINCT MEETING_DATE) as UNIQUE_MEETING_DATES,
    MIN(MEETING_DATE) as EARLIEST_MEETING,
    MAX(MEETING_DATE) as LATEST_MEETING,
    MAX(DOWNLOAD_TIMESTAMP) as LAST_DOWNLOAD
FROM ADVANCED_ANALYTICS.UNSTRUCTURED.MINUTES_METADATA;

## 🤖 AI-Powered Document Processing

Now let's use Snowflake's AI capabilities to parse the PDF documents and extract structured content.

We are going to use the [AI_PARSE_DOCUMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_parse_document) function. AI_PARSE_DOCUMENT is a Cortex AI SQL function that extracts text, data, and layout elements from documents with high fidelity.

The AI_PARSE_DOCUMENT function offers two modes for processing PDF documents:

* LAYOUT mode is the preferred choice for most use cases, especially for complex documents. It’s specifically optimized for extracting text and layout elements like tables, making it the best option for building knowledge bases, optimizing retrieval systems, and enhancing AI based applications.

* OCR mode is recommended for quick, high-quality text extraction from documents such as manuals, agreements or contracts, product detail pages, insurance policies and claims


In [ ]:
--let's see what this looks like on one document
SELECT AI_PARSE_DOCUMENT(TO_FILE('@ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES', 'bag121024_minutes.pdf'), { 'mode': 'LAYOUT', 'page_split': true }) AS parsed_document;

In [ ]:
--let's process all documents
CREATE TABLE IF NOT EXISTS ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES_PARSED
AS
WITH parsed_doc AS (
  SELECT 
    relative_path file_name,
    AI_PARSE_DOCUMENT(TO_FILE('@ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES', relative_path), 
                      { 'mode': 'LAYOUT', 'page_split': true }
                      ) AS parsed_document
  FROM directory(@ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES)
)
SELECT 
  file_name,
  page.value:index::INT +1 AS page_number,
  page.value:content::STRING AS page_content
FROM parsed_doc,
LATERAL FLATTEN(input => parsed_document:pages) AS page
;

In [ ]:
--let's examine the extracted text
SELECT * FROM ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES_PARSED LIMIT 10;

### 📝 Text Chunking in preparation for setting up a Cortex Search Service

Break down the parsed documents into optimally-sized chunks for better search and retrieval performance using the [SPLIT_TEXT_RECURSIVE_CHARACTER](https://docs.snowflake.com/en/sql-reference/functions/split_text_recursive_character-snowflake-cortex) function. This function splits a string into shorter stings, recursively, for preprocessing text to be used with text embedding or search indexing functions and returns an array of text chunks.

Note the parameters in the SPLIT_TEXT_RECURSIVE_CHARACTER function. Cortex Search Service uses the first 512 tokens in search, which is approximately 2,000 characters. Setting up chunking with slight overlap ensures that each chunk maintains some context from the previous chunk, preventing important information from being lost at chunk boundaries



In [ ]:
CREATE OR REPLACE TABLE ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES_CHUNKED
AS
SELECT 
  document_name,
  coalesce(meeting_date,
              try_to_date(AI_EXTRACT(
                    text => document_name,
                    responseFormat => {
                        'meeting_date': 'Extract the date from this filename and return it in yyyy-mm-dd format. 
                                         Look for date patterns like MMDDYY or MMDDYYYY. 
                                         If you find a date like 120820, convert it to 2020-12-08.'
                    }
                ):response.meeting_date::varchar)
            ) meeting_date,
  document_url,
  page_number,
  chunk.value::STRING AS chunk_content
FROM ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES_PARSED p,
LATERAL FLATTEN(
  input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
    page_content, 
    'markdown',
    2000,
    150
  )
) AS chunk,
ADVANCED_ANALYTICS.UNSTRUCTURED.MINUTES_METADATA m
WHERE p.file_name=m.document_name;

### 🔍 [Cortex Search Service](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

Create a powerful semantic search service that enables natural language queries across all meeting minutes.


In [ ]:
import streamlit as st
st.image('rag.png')

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES
    ON chunk_content
    ATTRIBUTES meeting_date
    WAREHOUSE = adhoc
    TARGET_LAG = '30 days'
    AS (
    SELECT
        document_name,
        chunk_content,
        document_url,
        meeting_date
    FROM ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES_CHUNKED
    );


In [ ]:
--test the search service'a ability to retrieve context
SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'ADVANCED_ANALYTICS.UNSTRUCTURED.SFBOS_MINUTES',
        '{
            "query": "What is the total amount appropriated to the Human Services Agency (HSA) for CalFresh (SNAP) Benefits Backfill in Fiscal Year (FY) 2025-2026?",
            "columns": ["chunk_content", "meeting_date"],
            "limit": 10
        }'
    )
)['results'] as results;

In [ ]:
# display the result
import streamlit as st
import pandas as pd

# Convert the previous cell output to pandas
df_pandas = cell14.to_pandas()  # If cell1 is a Snowpark DataFrame

# If you want to display the entire DataFrame as text
text_output = df_pandas.to_string()
st.text_area("DataFrame Output:", value=text_output, height=400)

---
### What We've Accomplished

✅ **Automated Data Collection**: Web scraping with intelligent date extraction from filenames  
✅ **Document Processing**: AI-powered PDF parsing using Snowflake's native capabilities  
✅ **Intelligent Search**: Cortex Search service for semantic document retrieval  

### Next Steps for Your Implementation

1. [**Set up Cortex Agent**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-manage): Use the cortex search service we created as an agent tool
2. **Query the data in** [**Snowflake Intelligence**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-intelligence)
3. **Set up** [**integration with MS Teams and Copilot**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-teams-integration) (work with your Azure team on setup)
